In [35]:
from dotenv import load_dotenv
load_dotenv()

True

In [36]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [37]:
loader = PyPDFLoader("../data/medical_report.pdf")
doc =loader.load()

In [38]:
spillter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap= 100)

In [39]:
spillter_doc = spillter.split_documents(doc)

In [40]:
len(spillter_doc)

25

In [41]:
embading =GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vector_store = InMemoryVectorStore.from_documents(
    documents=spillter_doc,
    embedding=embading
)

In [42]:
recor = vector_store.similarity_search("patient name")
recor[0].page_content

'Report Status    \nFemale\n27 Years:\n:\n:\n:\nAge\nGender\nReported        \nP\n9/7/2025   4:56:00PM\nDR NITIN NAHAR\n474764803\nMs. NIKITA  CHUDHARY:\n:\n:\n:\n:\nName        \nLab No.    \nRef By \nCollected       \nA/c Status \n10/7/2025  6:31:50PM\nFinal\nCollected at            : Processed at             :BHOPAL CC-82\nMr Rachel V John Pata So Vitus John Mig 26 \nGraund,Indrapuri, Phone: 8770817968\n \nLPL - Bhopal Lab II\nPlot No.05, Mandakini Housing Society, Near \nApurti Shopping Mall, Kolar Main Road, \nBhopal, M.P. -  462042\nTest Report      \nTest Name Results Units Bio. Ref. Interval\n 0.02 - 0.10 Basophils thou/mm30.04\nPlatelet Count  150.00 - 410.00 thou/mm3255\nMean Platelet Volume  6.5 - 12.0 fL9.1\nE.S.R.  0.00 - 20.00 mm/hr6\nComment\nIn anaemic conditions Mentzer index is used to differentiate Iron Deficiency Anaemia from Beta- Thalassemia \ntrait. If Mentzer Index value is > 13, there is probability of Iron Deficiency Anaemia. A value < 13 indicates likelihood'

In [43]:
##agent llm tools promts

In [49]:
@tool
def retever_tool(query:str):
    """
        this tools can help youto retrive the relavent data of pdf document and these document have detailabout medical eports


    """

    docs = vector_store.similarity_search(query=query,k=4)
    print(len(docs),docs[0].page_content)
    contex= ""
    for doc in docs:
        contex= doc.page_content + " \n\n"


    return contex


In [50]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

In [51]:
system_prompt = """you are a help fulll assistance that answerquestion using retrving context always usethe `retever_tool` tool for question requiring external knowledge"""


In [52]:
agent = create_agent(model= llm,
                     tools= [retever_tool],
                     system_prompt=system_prompt)

In [53]:
query ="what is the name of patient ,and what is the name of docters"
response = agent.invoke({"messages":[{"role":"user","content":query}]})

4 NRL - Dr Lal PathLabs Ltd
Dr Sunanda
MD, Pathology
Sr. Consultant Pathologist - 
Hematology & Immunology                              
NRL - Dr Lal PathLabs Ltd
-------------------------------End of report --------------------------------
AHEEEHAPMKHJBKHNLKIHBLLCBILLJCECCJLCIKPLPKEDFBFAPPAHEEEHA
BNFFFNBPAPBOACIGFGELNGAPAOAHFHAKAKNOBCJKBLEKMCJGNPBNFFFNB
CIEGCAFJLPNBKNNOPFIAAJHFJGDEHEDFLKPHENFLMLKIOEFLBKHDEHANP
DJECMEFNBMHIEDINMIMELFMFAFPLBLAFJIFFOAPAKCLJPDNLIJFBKEMEK
KDCILHFJONFFDIAJMBICKPJDCENJKPCHKKMBACOIPLKGKPNGKFFFOIOND
FGCBBEFNNICGDKHKILIGCIPBADHFOFAINCFCBKDKBLIKOMNALNEJOGEKD
DPIMIJFNELODDJHIEKMPCGFHIHAFJBALPPMPBLMIKJCHKJNKIJNFMDILD
NJJJAMFMAKNKPELJGANHDOMILLBMFMBHIFCCAMNNIIKNOPNKBNFMBHILL
NKIHJAFMDKOGFAAMFNOIHMIKHEJFEKFBJFFKBKOFOOBIOCNKDJPHNEKLJ
NMDBPJFCEHNJJDCLCDBFGPNFIDFKPNJHKEPLELPLNKCKIGEOCIGKHMEJP
ICIKOLFLGKCBMMCMPNBDAJEEINMEFCBBJMFEBFOEONDMKPFKEPCIGKAMG
ACICGJFCPLDDJOEMHHGHBPDEIEOPPFOFAOHPBLNOOLJBIKNJPKONHDICL
MNNNNNEHKHANBCCJOCFLHILHJBAHFHADLKPFCMNLMKJCLFNOAHFHAHIKL
4 MNN

In [54]:
result = response["messages"][-1].content

In [55]:
print(result)

[{'type': 'text', 'text': 'Based on the medical report:\n\n* **Patient Name:** Ms. NIKITA CHUDHARY\n* **Referring Doctor:** Dr. NITIN NAHAR\n\n*(Note: The report is also verified/signed by laboratory doctors/pathologists: Dr. Beena Chandrasekhar, Dr. Nagarjun Sai Jaine, and Dr. Sunanda).*', 'extras': {'signature': 'Es0ECsoEARFNMg8Z2ITfZafvYaY3d9gCCLKU2A1pgIObstc+jmcdEgj3625F4BJoWjoSYQEWu+aJ2j9q5G9eGVUIn1hQC8lC4S3T+v7w9pu1mlOfE5gKJ+eWnGk945Wry2/Fa/y6D/OjiDXaAWW9bIpPe8riYdqiC97hzc1uzU7m7BQnd8ldiWvIk/HGiYLiafg4rPZUHHBMolyP380gE4t0Lkdoz88zoM0z3HPbgOKmGX+xmW+Vksi7b65NXmqSjWr3ugwlxw5iQEzkMD4OWC34MdY0LLFM3QiGV7hRiSeU6dbO2bSZYioEzPyqynuMaTtmOM63US1RhdO3Dl/5sbjxjZlTqyl311HWwZUbzEzI0my7Ta00WN4v90jZuFvMd+C9x/CFnf2AmWBPpA01mtjtMLvqegL8CLBRKTFVeMz1QAebdDFeVwZVzIkWPjfqx/85wVVA6CpyuVvJLJ++TDDnyrT/HN5jQSV8AxERNTQUFP9+aCi4J87dMRomnpgEPwMeh8UuECirWdtodSfe2nZN5IuderaEj1eEjiGnaJLvr0SJlX02ffo6BV09RF4Bbx9ex3P9NDKlMoVv7QlPGDvJwWwqL6nPmXZsPS5Qrnx8Ki7L6qiTOjqkToImR92c4CVge8b/x7Y8wnbeshJvt8byuaF0/NuiW7J8nTY1VGw